In [ ]:
# !pip install evaluate

In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments

In [ ]:
!kaggle datasets download -d nileshmalode1/samsum-dataset-text-summarization

In [ ]:
!unzip samsum-dataset-text-summarization.zip

In [ ]:

train_df = pd.read_csv("samsum-train.csv")
test_df = pd.read_csv("samsum-test.csv")



In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
 # Duplicated rows

In [ ]:
dup_count = train_df.duplicated().sum()
print(f"Number of duplicated rows: {dup_count}\n")

In [ ]:
dup_count = test_df.duplicated().sum()
print(f"Number of duplicated rows: {dup_count}\n")

In [ ]:
# Null values

In [ ]:
nulls = train_df.isnull().sum()
print("Null values per column:")
print(nulls, "\n")

In [ ]:
train_df = train_df.dropna()
train_df.isnull().sum()


In [ ]:
nulls = test_df.isnull().sum()
print("Null values per column:")
print(nulls, "\n")

In [ ]:
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)


In [ ]:
print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

# Data Preprocessing

In [ ]:
def clean_text(text):
  text = re.sub(r'\r\n', ' ', text)
  text = re.sub(r'\s+', ' ', text)
  text = re.sub(r'<.*?>', '', text)
  text = text.strip().lower()
  return text



In [ ]:
# train_df['dialogue'][0]

In [ ]:
train_df['dialogue'] = train_df['dialogue'].apply(clean_text)
train_df['summary'] = train_df['summary'].apply(clean_text)




In [ ]:
val_df['dialogue'] = val_df['dialogue'].apply(clean_text)
val_df['summary'] = val_df['summary'].apply(clean_text)


In [ ]:
val_df

# Tokenization

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("google-t5/t5-small")

In [ ]:
# convert Pandas df to hugging face aataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)


In [ ]:
def tokenization_preprocessing(df):
  inputs = tokenizer(df['dialogue'], padding='max_length', truncation=True, max_length= 512)
  targets = tokenizer(df['summary'], padding='max_length', truncation=True, max_length= 200)

  inputs['labels'] = targets['input_ids']
  return inputs



In [ ]:
train_dataset = train_dataset.map(tokenization_preprocessing)
val_dataset = val_dataset.map(tokenization_preprocessing)

In [ ]:
train_dataset[0]

# Fine Tuning Model

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("google-t5/t5-small")



In [ ]:
# training arguments

training_args = TrainingArguments(
     output_dir="./results",
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    save_steps=500,
    eval_steps=50,
    eval_strategy="epoch",
    fp16=False,
     optim="adamw_torch",

)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)  # Move to GPU if available


In [ ]:
# save the model

model.save_pretrained("./summary_model/model")
tokenizer.save_pretrained("./summary_model/tokenizer")

In [ ]:
import shutil

folder_path = "./summary_model"

# Output zip file name
zip_path = "summary_model.zip"

# Create a zip file
shutil.make_archive("summary_model", 'zip', folder_path)

print("Folder zipped successfully!")

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("./summary_model/model")
tokenizer = T5Tokenizer.from_pretrained("./summary_model/tokenizer")



In [ ]:
device = model.device



def summarization(text):
  text = clean_text(text)
  inputs = tokenizer(text, return_tensors='pt', padding = True, max_length=512, truncation=True)

  inputs = {key: value.to(device) for key,value in inputs.items()}


  # generate summary

  outputs = model.generate(
     inputs['input_ids'],
     attention_mask=inputs['attention_mask'],
     max_length = 200,
     num_beams = 5,
     early_stopping = True,
     min_length=50,
     no_repeat_ngram_size=3
  )

  summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return summary

In [ ]:
text_example = """
The quality of your data directly impacts the accuracy of your analysis and model performance. Why? Because raw data often contains inconsistencies, errors, and irrelevant information that can distort results and lead to flawed insights. Data preprocessing is a way to mitigate this problem. Namely, it is the process of transforming raw data into a clean, structured format.

In this blog post, I will cover:

What data preprocessing?
Steps in data preprocessing
Techniques for data preprocessing with examples
Tools for data preprocessing
Best practices for data preprocessing
Let’s get into it!

What is Data Preprocessing?
Data preprocessing is a key aspect of data preparation. It refers to any processing applied to raw data to ready it for further analysis or processing tasks.

Traditionally, data preprocessing has been an essential preliminary step in data analysis. However, more recently, these techniques have been adapted to train machine learning and AI models and make inferences from them.

Thus, data preprocessing may be defined as the process of converting raw data into a format that can be processed more efficiently and accurately in tasks such as:

Data analysis
Machine learning
Data science
AI
Become a Data Engineer
Build Python skills to become a professional data engineer.
Steps in Data Preprocessing
Data preprocessing involves several steps, each addressing specific challenges related to data quality, structure, and relevance.

Let’s take a look at these key steps, which generally go in the following order:

Step 1: Data cleaning
Data cleaning is the process of identifying and correcting errors or inconsistencies in the data to ensure it is accurate and complete. The objective is to address issues that can distort analysis or model performance.

For example:

Handling missing values: Using strategies like mean/mode imputation, deletion, or predictive models to fill in or remove missing data.
Removing duplicates: Eliminating duplicate records to ensure each entry is unique and relevant.
Correcting inconsistent formats: Standardizing formats (e.g., date formats, string cases) to maintain consistency.
Here’s how it looks in Python:
"""

# Generate summary
summary_result = summarization(text_example)
# print("Original Text:\n", text_example)
print("\nGenerated Summary:\n", summary_result)

In [ ]:
# !pip install bert-score

In [ ]:
# !pip install evaluate

# !pip install rouge_score

In [ ]:
import evaluate
from bert_score import score

In [ ]:
rouge = evaluate.load("rouge")
print("Evaluate library loaded successfully!")

In [ ]:

def evaluate_summaries(predictions, references):
    #  ROUGE
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=predictions, references=references)

    print("ROUGE scores:")
    for key, value in rouge_results.items():
        print(f"{key}: {value:.4f}")

    # BERTScore
    P, R, F1 = score(predictions, references, lang="en", verbose=True)
    print("\nBERTScore F1:", F1.mean().item())

In [ ]:
# Generate predictions
test_texts = val_df['dialogue'].tolist()
reference_summaries = val_df['summary'].tolist()

predictions = [summarization(text) for text in test_texts[:10]]
references = reference_summaries[:10]

# Evaluate
evaluate_summaries(predictions, references)